# Pipeline — Paso 6: Limpieza del dataset

Aplica filtros de calidad (confianza, duración, keypoints válidos) y genera dataset_limpio.csv y dataset_descartados.csv.

In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils.config import DATA_DIR

CSV_ENTRADA    = os.path.join(DATA_DIR, "datos", "dataset_anotaciones_unificado.csv")
CSV_LIMPIO     = os.path.join(DATA_DIR, "anotaciones", "label_studio", "dataset_limpio.csv")
CSV_DESC       = os.path.join(DATA_DIR, "datos", "dataset_descartados.csv")
CSV_COMP35     = os.path.join(DATA_DIR, "datos", "dataset_comparativa_35.csv")

# Umbrales de calidad (solo para Label Studio)
CONF_MIN = 0.3
KP_MIN   = 0.5
DUR_MIN  = 0.3


def razon_descarte(row) -> str:
    razones = []
    if pd.notna(row.get("confianza")) and row["confianza"] < CONF_MIN:
        razones.append(f"conf={row['confianza']:.2f}<{CONF_MIN}")
    if pd.notna(row.get("keypoints_validos")) and row["keypoints_validos"] < KP_MIN:
        razones.append(f"kp={row['keypoints_validos']:.2f}<{KP_MIN}")
    if row["duracion_seg_etiqueta"] < DUR_MIN:
        razones.append(f"dur={row['duracion_seg_etiqueta']:.2f}<{DUR_MIN}")
    if pd.isna(row.get("tiempo_inicio")) or pd.isna(row.get("tiempo_fin")):
        razones.append("sin_tiempos")
    return " | ".join(razones) if razones else "ok"


def filtrar_ls(df: pd.DataFrame) -> pd.Series:
    """
    Filtros de calidad aplicados solo a registros de Label Studio:
      - duracion_seg_etiqueta >= DUR_MIN   (0.3 s)
      - tiempo_inicio / tiempo_fin no nulos
      - confianza >= CONF_MIN             (0.3)  — solo si la columna tiene datos
      - keypoints_validos >= KP_MIN       (0.5)  — solo si la columna tiene datos
        (requiere que 04b_completar_metricas.py haya sido ejecutado)
    """
    mask = df["duracion_seg_etiqueta"] >= DUR_MIN
    mask &= df["tiempo_inicio"].notna() & df["tiempo_fin"].notna()
    # Confianza: filtrar solo si la columna tiene datos
    if df["confianza"].notna().any():
        mask &= df["confianza"].fillna(CONF_MIN) >= CONF_MIN
    # keypoints_validos: filtrar solo si 04b ya la pobló
    if "keypoints_validos" in df.columns and df["keypoints_validos"].notna().any():
        mask &= df["keypoints_validos"].fillna(1.0) >= KP_MIN
    return mask


def main():
    if not os.path.exists(CSV_ENTRADA):
        print(f"[ERROR] No existe {CSV_ENTRADA}")
        print("  Ejecuta primero el notebook pipeline/05_analizar_dataset.ipynb")
        return

    df = pd.read_csv(CSV_ENTRADA)
    herramientas = df["herramienta"].unique()
    print(f"Dataset unificado: {len(df)} registros | herramientas: {list(herramientas)}")

    # ── 1. Filtrado principal: solo Label Studio ──────────────────────────────
    df_ls   = df[df["herramienta"] == "label_studio"].copy()
    df_rest = df[df["herramienta"] != "label_studio"].copy()   # cvat + elan: sin filtro

    mask_ls        = filtrar_ls(df_ls)
    df_ls_limpio   = df_ls[mask_ls].copy()
    df_ls_desc     = df_ls[~mask_ls].copy()
    df_ls_desc["razon_descarte"] = df_ls_desc.apply(razon_descarte, axis=1)

    # CVAT y ELAN se conservan completos (anotaciones manuales de referencia)
    df_limpio      = pd.concat([df_ls_limpio, df_rest], ignore_index=True)
    df_descartados = df_ls_desc.copy()

    print(f"\n{'='*50}")
    print(f"{'Herramienta':<20} {'Total':>6} {'Limpios':>8} {'Descartados':>12}")
    print("-" * 50)
    for h in list(herramientas):
        tot  = len(df[df["herramienta"] == h])
        limp = len(df_limpio[df_limpio["herramienta"] == h])
        desc = tot - limp
        print(f"{h:<20} {tot:>6} {limp:>8} {desc:>12}")
    print(f"{'TOTAL':<20} {len(df):>6} {len(df_limpio):>8} {len(df_descartados):>12}")

    df_limpio.to_csv(CSV_LIMPIO, index=False, encoding="utf-8-sig")
    df_descartados.to_csv(CSV_DESC, index=False, encoding="utf-8-sig")
    print(f"\nGuardado: {CSV_LIMPIO}")
    print(f"Guardado: {CSV_DESC}")

    # ── 2. Subconjunto comparativo: mismos N videos × todas las herramientas ────
    # Referencia = INTERSECCION de pares (video_id, glosa) entre CVAT y ELAN.

    pares_manuales = {}
    for h in ["cvat", "elan"]:
        if h in herramientas:
            df_h = df[df["herramienta"] == h]
            pares_manuales[h] = set(zip(df_h["video_id"], df_h["glosa"]))

    if len(pares_manuales) >= 2:
        # Interseccion: solo videos que tienen anotacion en TODAS las herramientas manuales
        ref_pares = pares_manuales["cvat"] & pares_manuales["elan"]
    elif len(pares_manuales) == 1:
        ref_pares = next(iter(pares_manuales.values()))
    else:
        ref_pares = None

    if ref_pares:
        n_ref = len(ref_pares)

        # Filtrar cada herramienta al subconjunto comun
        def en_ref(df_h):
            return df_h[df_h.apply(
                lambda r: (r["video_id"], r["glosa"]) in ref_pares, axis=1)].copy()

        # LS: si hay duplicados (mismo video_id+glosa), conservar el de mayor confianza
        df_ls_comp = en_ref(df_ls)
        if len(df_ls_comp) > n_ref:
            df_ls_comp = (df_ls_comp
                          .sort_values("confianza", ascending=False)
                          .drop_duplicates(subset=["video_id", "glosa"])
                          .reset_index(drop=True))

        # CVAT y ELAN: filtrar tambien a la interseccion
        partes_comp = [df_ls_comp]
        for h in ["cvat", "elan"]:
            if h in herramientas:
                partes_comp.append(en_ref(df[df["herramienta"] == h]))

        df_comp = pd.concat(partes_comp, ignore_index=True)

        print(f"\n{'='*50}")
        print(f"SUBCONJUNTO COMPARATIVO (mismos {n_ref} videos, interseccion CVAT+ELAN)")
        print(f"{'='*50}")
        for h in list(herramientas):
            n     = len(df_comp[df_comp["herramienta"] == h])
            falta = n_ref - n
            nota  = f"  [!] faltan {falta} en LS" if falta > 0 and h == "label_studio" else ""
            print(f"  {h:<20} {n:>3} registros{nota}")
        print(f"  {'TOTAL':<20} {len(df_comp):>3} registros")

        df_comp.to_csv(CSV_COMP35, index=False, encoding="utf-8-sig")
        print(f"\nGuardado: {CSV_COMP35}")

main()


[ERROR] No existe C:\Users\dell\Documents\TIC\Codigo_TIC\datos\dataset_anotaciones_unificado.csv
  Ejecuta primero el notebook pipeline/05_analizar_dataset.ipynb


C:\Users\dell\AppData\Local\Temp\ipykernel_7756\1069956323.py:17: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
